# Regressão linear múltipla

**Objetivo:** ajustar um hiperplano a vários preditores de uma vez — pela equação normal com NumPy e com o scikit-learn — e diagnosticar colinearidade com o VIF, tudo com laços à mostra.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## 1. Todos os preditores

O conjunto diabetes tem 10 preditores clínicos. Vamos usá-los todos.

In [ ]:
from sklearn.datasets import load_diabetes

dados = load_diabetes(as_frame=True)
X = dados.data.values      # (n, 10)
y = dados.target.values
nomes = list(dados.data.columns)
print("X tem forma (n, p) =", X.shape)
print("preditores:", nomes)

## 2. A equação normal

Empilhando uma coluna de $1$ para o intercepto, a solução de mínimos quadrados é $\boldsymbol\theta = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$. Na prática resolvemos o sistema linear (mais estável do que inverter a matriz).

In [ ]:
n = X.shape[0]
X_intercepto = np.hstack([np.ones((n, 1)), X])   # coluna de 1s + preditores
A = X_intercepto.T @ X_intercepto
b = X_intercepto.T @ y
theta = np.linalg.solve(A, b)                    # resolve A theta = b
print("theta0 (intercepto):", round(theta[0], 2))
for nome, coef in zip(nomes, theta[1:]):
    print(nome.ljust(5), "->", round(coef, 1))

## 3. Conferindo com o scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()
modelo.fit(X, y)
print("intercepto bate?", np.isclose(modelo.intercept_, theta[0]))
print("coeficientes batem?", np.allclose(modelo.coef_, theta[1:]))
print("R2 do modelo completo:", round(modelo.score(X, y), 3))

## 4. Colinearidade: o VIF de cada preditor

Para cada preditor, regredimos ele contra **todos os outros** e medimos o $R^2$; o VIF é $1/(1-R^2)$. VIF alto = preditor redundante. Fazemos num laço explícito, um preditor por vez.

In [ ]:
vifs = []
for j in range(X.shape[1]):
    outros = [c for c in range(X.shape[1]) if c != j]
    reg = LinearRegression().fit(X[:, outros], X[:, j])
    r2_j = reg.score(X[:, outros], X[:, j])
    vif_j = 1.0 / (1.0 - r2_j)
    vifs.append(vif_j)
    print(nomes[j].ljust(5), "R2 =", round(r2_j, 3), "| VIF =", round(vif_j, 2))

In [ ]:
figura = go.Figure(go.Bar(x=nomes, y=vifs, marker_color=VERMELHO,
                          text=[round(v, 1) for v in vifs], textposition="outside"))
figura.add_hline(y=5, line_dash="dash", line_color=TINTA,
                 annotation_text="atencao acima de 5")
figura.update_layout(title="Fator de inflacao da variancia (VIF) por preditor",
                     yaxis_title="VIF", height=360, margin=dict(l=10, r=10, t=50, b=10))
figura.show()
print("preditores com VIF > 5 (candidatos a redundancia):",
      [nomes[j] for j in range(len(vifs)) if vifs[j] > 5])

## Exercício

Os preditores `s1`–`s6` são medidas de sangue, algumas muito correlacionadas. Olhe os VIF: quais formam o par mais redundante? O que aconteceria com a interpretação dos coeficientes deles?

In [ ]:
# @title Solução (clique para revelar)
ordem = np.argsort(vifs)[::-1]
print("preditores por VIF (maior primeiro):")
for j in ordem[:4]:
    print("  ", nomes[j], "VIF =", round(vifs[j], 1))
# Os de VIF mais alto carregam quase a mesma informacao; seus coeficientes
# individuais ficam instaveis (mudam muito com pequenas variacoes nos dados),
# entao nao devem ser lidos isoladamente. A regularizacao (proximo topico) ajuda.